# Chương 9: Thực hành - Mạng nơ-ron đa tầng truyền thẳng (Multi-Layer Perceptron from Scratch)

**Mục tiêu bài thực hành:**
1. **Giải quyết bài toán XOR kinh điển:** Xây dựng mạng MLP nhỏ giải quyết thành công hàm XOR (nơi mô hình tuyến tính thất bại hoàn toàn).
2. **Tự cài đặt Mạng MLP tổng quát (`ModularMLP`) từ đầu:** Viết trọn vẹn thuật toán Lan truyền tiến (Forward Pass) và Lan truyền ngược (Backpropagation) chỉ bằng `NumPy`.
3. **Chinh phục Dữ liệu Xoắn ốc Phi tuyến (Two Spirals Dataset):** Huấn luyện mạng MLP phân loại thành công 2 vòng xoắn ốc phức tạp và vẽ ranh giới quyết định uốn lượn.
4. **Kiểm chứng Hiện tượng Triệt tiêu Gradient (Vanishing Gradient):** So sánh trực tiếp độ lớn gradient giữa mạng sâu dùng Sigmoid vs mạng sâu dùng ReLU.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Thiết lập hạt giống ngẫu nhiên
np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. Hóa giải Bài toán XOR Kinh điển bằng Mạng MLP

Dữ liệu: $\mathbf{X} = [[0, 0], [0, 1], [1, 0], [1, 1]]$, Nhãn: $\mathbf{y} = [0, 1, 1, 0]^T$.  
Kiến trúc mạng: Đầu vào (2 nơ-ron) $\to$ Tầng ẩn (4 nơ-ron ReLU) $\to$ Đầu ra (1 nơ-ron Sigmoid).

In [ ]:
X_xor = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
y_xor = np.array([[0], [1], [1], [0]], dtype=float)

# Khởi tạo trọng số ngẫu nhiên
W1 = np.random.randn(2, 4) * 0.8
b1 = np.zeros((1, 4))
W2 = np.random.randn(4, 1) * 0.8
b2 = np.zeros((1, 1))

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -250, 250)))

lr = 0.15
losses = []

for epoch in range(1200):
    # --- Lan truyền tiến (Forward Pass) ---
    z1 = X_xor @ W1 + b1
    h1 = np.maximum(0, z1) # ReLU
    z2 = h1 @ W2 + b2
    y_hat = sigmoid(z2)
    
    # Binary Cross-Entropy Loss
    eps = 1e-15
    loss = -np.mean(y_xor * np.log(y_hat + eps) + (1 - y_xor) * np.log(1 - y_hat + eps))
    losses.append(loss)
    
    # --- Lan truyền ngược (Backpropagation) ---
    delta2 = y_hat - y_xor                                 # N x 1
    dW2 = h1.T @ delta2 / len(X_xor)                      # 4 x 1
    db2 = np.sum(delta2, axis=0, keepdims=True) / len(X_xor)
    
    delta1 = (delta2 @ W2.T) * (z1 > 0)                   # Đạo hàm ReLU: 1 nếu z1 > 0, 0 nếu ngược lại
    dW1 = X_xor.T @ delta1 / len(X_xor)                    # 2 x 4
    db1 = np.sum(delta1, axis=0, keepdims=True) / len(X_xor)
    
    # Cập nhật trọng số
    W2 -= lr * dW2
    b2 -= lr * db2
    W1 -= lr * dW1
    b1 -= lr * db1

print("=== KẾT QUẢ DỰ ĐOÁN HÀM XOR SAU HUẤN LUYỆN ===")
for x_in, target, pred in zip(X_xor, y_xor, y_hat):
    print(f"Đầu vào: {x_in.astype(int)} -> Thực tế: {int(target[0])} | Dự đoán xác suất: {pred[0]:.4f} (Đoán: {int(pred[0] >= 0.5)})")

## 2. Cài đặt Lớp Mạng Nơ-ron MLP Tổng quát (`ModularMLP` from scratch)

In [ ]:
class ModularMLP:
    def __init__(self, layer_dims, activation='relu'):
        """
        layer_dims: Danh sách số lượng nơ-ron các tầng, ví dụ: [2, 32, 16, 1]
        """
        self.layer_dims = layer_dims
        self.activation = activation
        self.params = {}
        self.L = len(layer_dims) - 1
        
        # Khởi tạo He (Kaiming) initialization cho ReLU
        for l in range(1, self.L + 1):
            scale = np.sqrt(2.0 / layer_dims[l-1]) if activation == 'relu' else np.sqrt(1.0 / layer_dims[l-1])
            self.params[f'W{l}'] = np.random.randn(layer_dims[l-1], layer_dims[l]) * scale
            self.params[f'b{l}'] = np.zeros((1, layer_dims[l]))
            
    def _act(self, z):
        if self.activation == 'relu':
            return np.maximum(0, z)
        elif self.activation == 'tanh':
            return np.tanh(z)
        elif self.activation == 'sigmoid':
            return 1.0 / (1.0 + np.exp(-np.clip(z, -250, 250)))
            
    def _act_prime(self, z):
        if self.activation == 'relu':
            return (z > 0).astype(float)
        elif self.activation == 'tanh':
            return 1.0 - np.tanh(z)**2
        elif self.activation == 'sigmoid':
            s = self._act(z)
            return s * (1.0 - s)
            
    def forward(self, X):
        cache = {'H0': X}
        H = X
        for l in range(1, self.L):
            Z = H @ self.params[f'W{l}'] + self.params[f'b{l}']
            H = self._act(Z)
            cache[f'Z{l}'] = Z
            cache[f'H{l}'] = H
            
        # Tầng cuối dùng Sigmoid phân loại nhị phân
        ZL = H @ self.params[f'W{self.L}'] + self.params[f'b{self.L}']
        HL = 1.0 / (1.0 + np.exp(-np.clip(ZL, -250, 250)))
        cache[f'Z{self.L}'] = ZL
        cache[f'H{self.L}'] = HL
        return HL, cache
        
    def backward(self, Y, cache):
        grads = {}
        N = Y.shape[0]
        
        # Tầng đầu ra: delta_L = Y_hat - Y
        delta = cache[f'H{self.L}'] - Y
        grads[f'dW{self.L}'] = (cache[f'H{self.L-1}'].T @ delta) / N
        grads[f'db{self.L}'] = np.sum(delta, axis=0, keepdims=True) / N
        
        # Truyền ngược về các tầng ẩn
        for l in range(self.L - 1, 0, -1):
            delta = (delta @ self.params[f'W{l+1}'].T) * self._act_prime(cache[f'Z{l}'])
            grads[f'dW{l}'] = (cache[f'H{l-1}'].T @ delta) / N
            grads[f'db{l}'] = np.sum(delta, axis=0, keepdims=True) / N
            
        return grads
        
    def update_params(self, grads, lr):
        for l in range(1, self.L + 1):
            self.params[f'W{l}'] -= lr * grads[f'dW{l}']
            self.params[f'b{l}'] -= lr * grads[f'db{l}']

## 3. Thử thách trên Dữ liệu Xoắn ốc Phi tuyến (Two Spirals Challenge)

In [ ]:
def generate_spirals(n_points=250, noise=0.2):
    n = n_points // 2
    theta = np.sqrt(np.random.rand(n)) * 2 * np.pi
    r_a = 2 * theta + np.pi
    data_a = np.array([np.cos(theta) * r_a, np.sin(theta) * r_a]).T + np.random.randn(n, 2) * noise
    
    r_b = -2 * theta - np.pi
    data_b = np.array([np.cos(theta) * r_b, np.sin(theta) * r_b]).T + np.random.randn(n, 2) * noise
    
    X = np.vstack([data_a, data_b]) / 15.0 # Chuẩn hóa khoảng giá trị
    y = np.vstack([np.zeros((n, 1)), np.ones((n, 1))])
    return X, y

X_spiral, y_spiral = generate_spirals(n_points=400, noise=0.25)

# Huấn luyện mạng MLP cấu trúc: 2 -> 32 -> 16 -> 1 nơ-ron
model = ModularMLP(layer_dims=[2, 32, 16, 1], activation='relu')
loss_curve = []
epochs = 2000
lr = 0.2

for ep in range(epochs):
    y_pred, cache = model.forward(X_spiral)
    eps = 1e-15
    loss = -np.mean(y_spiral * np.log(y_pred + eps) + (1 - y_spiral) * np.log(1 - y_pred + eps))
    loss_curve.append(loss)
    
    grads = model.backward(y_spiral, cache)
    model.update_params(grads, lr)

final_acc = np.mean((y_pred >= 0.5) == y_spiral)
print(f"[MLP trên Spiral Dataset] Độ chính xác đạt được: {final_acc * 100:.2f}% | Loss cuối: {loss_curve[-1]:.4f}")

## 4. Trực quan hóa Ranh giới Quyết định Phi tuyến tính (Spiral Decision Boundary)

In [ ]:
# Tạo lưới phẳng 2D
x_span = np.linspace(-1.3, 1.3, 200)
y_span = np.linspace(-1.3, 1.3, 200)
xx, yy = np.meshgrid(x_span, y_span)
grid_data = np.c_[xx.ravel(), yy.ravel()]

probs, _ = model.forward(grid_data)
probs = probs.reshape(xx.shape)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Đồ thị hàm mất mát
ax1.plot(loss_curve, color='purple', linewidth=2)
ax1.set_title("Đường cong Mất mát (BCE Loss) qua 2000 Epochs", fontsize=12, fontweight='bold')
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")

# Đồ thị ranh giới phân loại xoắn ốc
contour = ax2.contourf(xx, yy, probs, levels=20, cmap='coolwarm', alpha=0.6)
ax2.contour(xx, yy, probs, levels=[0.5], colors='black', linewidths=2.5)
ax2.scatter(X_spiral[y_spiral.squeeze() == 0, 0], X_spiral[y_spiral.squeeze() == 0, 1], c='#2980b9', edgecolors='k', label='Lớp 0')
ax2.scatter(X_spiral[y_spiral.squeeze() == 1, 0], X_spiral[y_spiral.squeeze() == 1, 1], c='#c0392b', edgecolors='k', label='Lớp 1')
ax2.set_title(f"Ranh giới Quyết định Xoắn ốc của MLP (Acc = {final_acc*100:.1f}%)", fontsize=12, fontweight='bold')
ax2.set_xlim(-1.3, 1.3)
ax2.set_ylim(-1.3, 1.3)
ax2.legend(loc='upper right')

plt.suptitle("Năng lực Bẻ cong Không gian và Khớp Dữ liệu Phi tuyến của Mạng Nơ-ron MLP", fontsize=14, y=0.98)
plt.tight_layout()
plt.show()

## 5. Minh họa Hiện tượng Triệt tiêu Gradient (Vanishing Gradient): Sigmoid vs. ReLU

Huấn luyện mạng nơ-ron sâu 6 tầng: `[2, 16, 16, 16, 16, 1]`. Đo lường độ lớn chuẩn gradient $\|\nabla_{\mathbf{W}^{(1)}} \mathcal{L}\|_2$ tại tầng đầu tiên.

In [ ]:
deep_dims = [2, 16, 16, 16, 16, 1]
model_sigmoid = ModularMLP(deep_dims, activation='sigmoid')
model_relu = ModularMLP(deep_dims, activation='relu')

grad_norms_sigmoid = []
grad_norms_relu = []

for ep in range(150):
    # Sigmoid forward & backward
    y_s, cache_s = model_sigmoid.forward(X_spiral)
    grads_s = model_sigmoid.backward(y_spiral, cache_s)
    grad_norms_sigmoid.append(np.linalg.norm(grads_s['dW1']))
    model_sigmoid.update_params(grads_s, lr=0.1)
    
    # ReLU forward & backward
    y_r, cache_r = model_relu.forward(X_spiral)
    grads_r = model_relu.backward(y_spiral, cache_r)
    grad_norms_relu.append(np.linalg.norm(grads_r['dW1']))
    model_relu.update_params(grads_r, lr=0.1)

plt.figure(figsize=(11, 4.5))
plt.plot(grad_norms_relu, label='Mạng sâu ReLU (Duy trì dòng chảy gradient khỏe mạnh)', color='#27ae60', linewidth=2)
plt.plot(grad_norms_sigmoid, label='Mạng sâu Sigmoid (Gradient tầng đầu bị triệt tiêu gần 0)', color='#c0392b', linewidth=2)
plt.title("Đo lường Độ lớn Gradient tại Tầng 1: Sigmoid (Bị triệt tiêu) vs. ReLU", fontsize=12, fontweight='bold')
plt.xlabel("Epoch")
plt.ylabel("Chuẩn Frobenius ||dW1||")
plt.yscale('log')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

## 6. Tổng kết bài học thực hành

1. **Hàm kích hoạt phi tuyến:** Là yếu tố quyết định cho phép mạng nơ-ron vượt qua giới hạn của mô hình tuyến tính để giải quyết bài toán XOR và bài toán xoắn ốc.
2. **Lan truyền ngược (Backpropagation):** Bản chất là việc áp dụng đệ quy quy tắc chuỗi giải tích đa biến để tính đạo hàm chính xác với chi phí tính toán tối ưu $\mathcal{O}(\text{số kết nối})$.
3. **ReLU:** Khắc phục triệt để hiện tượng triệt tiêu gradient của hàm Sigmoid/Tanh, là phát minh bản lề đưa ngành Học sâu vươn lên thống trị AI hiện đại.